# Environment

The book runs on the Neurodesk GPU server (NVIDIA A40). Nothing is installed here: FreeSurfer
comes from the Neurodesk module, the Python packages (PyTorch included) from the `lfbook`
environment. The cells
below check that the tools and the input data are in place; later chapters assume they are.

In [2]:
import sys
from pathlib import Path

from common import P, OLD_RAW, RESULTS, THREADS, fs, sh

for k, v in P.items():
    print(f'{k:12s} {v}')
print(f"{'results now':12s} {RESULTS}")

data         /home/jovyan/neurodesktop-storage/neurodesk_upload
license      /home/jovyan/neurodesktop-storage/neurodesk_upload/freesurfer/license.txt
results      /home/jovyan/neurodesktop-storage/lfbook/results
work         /home/jovyan/neurodesktop-storage/lfbook/work
wmh_results  /home/jovyan/neurodesktop-storage/lfbook/results/old/seg/WMH_SynthSeg
results now  /home/jovyan/neurodesktop-storage/lfbook/results


**Resources.** CPU and memory limits of this Neurodesk session. SynthSeg and SynthMorph use
all CPUs.

In [3]:
mem = Path('/sys/fs/cgroup/memory.max').read_text().strip()
print('python', sys.executable)
print(f'{THREADS} CPUs | memory limit', mem if mem == 'max' else f'{int(mem) / 1e9:.0f} GB')

python /home/jovyan/conda-envs/lfbook/bin/python
8 CPUs | memory limit 32 GB


**GPU.** WMH-SynthSeg (Chapter 03) runs on the GPU.

In [ ]:
import torch

assert torch.cuda.is_available(), 'no GPU: start Neurodesk on the A40 server'
print('torch', torch.__version__)
sh('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader')

**FreeSurfer.** SynthSeg, SynthMorph and `mri_vol2vol` from the FreeSurfer 8.0.0 module.

In [4]:
assert P['license'].is_file(), f"FreeSurfer licence not found: {P['license']}"
fs('which mri_synthseg mri_synthmorph mri_vol2vol && echo FS_LICENSE=$FS_LICENSE')

/cvmfs/neurodesk.ardc.edu.au/containers/freesurfer_8.0.0_20260313/mri_synthseg
/cvmfs/neurodesk.ardc.edu.au/containers/freesurfer_8.0.0_20260313/mri_synthmorph
/cvmfs/neurodesk.ardc.edu.au/containers/freesurfer_8.0.0_20260313/mri_vol2vol
FS_LICENSE=/home/jovyan/neurodesktop-storage/neurodesk_upload/freesurfer/license.txt


**Input data.** Old dataset, localizers not counted.

In [5]:
def nii(root):
    return sorted(f for f in root.rglob('*.nii.gz') if 'localizer' not in f.name)


counts = {'old 3T': (len(nii(OLD_RAW / '3T')), 66), 'old 64mT': (len(nii(OLD_RAW / '64mT')), 61)}
for k, (n, expected) in counts.items():
    print(f'{k:9s} {n:3d} scans (expected {expected})')
assert all(n == e for n, e in counts.values()), 'scan count does not match'

old 3T     66 scans (expected 66)
old 64mT   61 scans (expected 61)
